This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).


In [ ]:
!pip install torch transformers sentencepiece peft accelerate --upgrade -q

## Text generation

### A brief history of sequence generation

### Training a mini-GPT

In [ ]:
# PyTorch: the Keras backend memory-tuning env vars are not needed.

In [ ]:
import pathlib
import urllib.request
import zipfile

# PyTorch: there is no keras.utils.get_file; download and extract the dataset
# with the standard library instead.
url = "https://hf.co/datasets/mattdangerw/mini-c4/resolve/main/mini-c4.zip"
zip_path = pathlib.Path("mini-c4.zip")
if not zip_path.exists():
    urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(".")
extract_dir = pathlib.Path(".") / "mini-c4"

In [0]:
with open(extract_dir / "shard0.txt", "r") as f:
    print(f.readline().replace("\\n", "\n")[:100])

In [ ]:
import urllib.request
import numpy as np
import sentencepiece as spm

# PyTorch: keras_hub's SentencePieceTokenizer is replaced by Google's
# sentencepiece library loading the same vocabulary.proto file.
vocabulary_file = "vocabulary.proto"
urllib.request.urlretrieve(
    "https://hf.co/mattdangerw/spiece/resolve/main/vocabulary.proto",
    vocabulary_file,
)
tokenizer = spm.SentencePieceProcessor(model_file=vocabulary_file)

In [ ]:
# PyTorch: sentencepiece uses .encode() / .decode() instead of callable tokenize.
tokenizer.encode("The quick brown fox.")

In [ ]:
tokenizer.decode([450, 4996, 17354, 1701, 29916, 29889])

In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader

# PyTorch: the tf.data pipeline becomes a streaming IterableDataset. We read each
# text file line by line, tokenize, append the <|endoftext|> id, and pack the
# token stream into fixed-length (sequence_length + 1) windows. Each window is
# split into (inputs, targets) shifted by one position, exactly like the book.
batch_size = 64
sequence_length = 256
suffix = tokenizer.piece_to_id("<|endoftext|>")

files = [str(file) for file in extract_dir.glob("*.txt")]


class C4Dataset(IterableDataset):
    def __init__(self, files, sequence_length):
        self.files = files
        self.window = sequence_length + 1

    def __iter__(self):
        buffer = []
        for filename in self.files:
            with open(filename, "r") as f:
                for line in f:
                    text = line.replace("\\n", "\n")
                    buffer.extend(tokenizer.encode(text))
                    buffer.append(suffix)
                    while len(buffer) >= self.window:
                        chunk = buffer[: self.window]
                        buffer = buffer[self.window :]
                        x = torch.tensor(chunk[:-1], dtype=torch.long)
                        y = torch.tensor(chunk[1:], dtype=torch.long)
                        yield x, y


dataset = C4Dataset(files, sequence_length)
ds = DataLoader(dataset, batch_size=batch_size, drop_last=True)

In [ ]:
import itertools

# PyTorch: there is no tf.data take/skip/repeat. We materialize fixed numbers of
# batches for validation and training from the streaming DataLoader. The book's
# .repeat() is handled by re-iterating the DataLoader each epoch.
num_batches = 58746
num_val_batches = 500
num_train_batches = num_batches - num_val_batches


def take(iterable, n):
    return list(itertools.islice(iterable, n))


batches = iter(ds)
val_batches = take(batches, num_val_batches)
train_loader = ds  # remaining batches are produced fresh each epoch

#### Building the model

In [ ]:
from torch import nn

# PyTorch: keras.Layer becomes an nn.Module. nn.MultiheadAttention replaces
# layers.MultiHeadAttention; we build a causal (triangular) attn_mask ourselves
# instead of use_causal_mask=True. batch_first=True keeps (batch, seq, dim).
class TransformerDecoder(nn.Module):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        self.self_attention = nn.MultiheadAttention(
            hidden_dim, num_heads, dropout=0.1, batch_first=True
        )
        self.self_attention_layernorm = nn.LayerNorm(hidden_dim)
        self.feed_forward_1 = nn.Linear(hidden_dim, intermediate_dim)
        self.feed_forward_2 = nn.Linear(intermediate_dim, hidden_dim)
        self.feed_forward_layernorm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(0.1)
        self.relu = nn.ReLU()

    def forward(self, inputs):
        seq_len = inputs.shape[1]
        # PyTorch: float causal mask (-inf above the diagonal) blocks attending
        # to future positions.
        causal_mask = torch.triu(
            torch.full((seq_len, seq_len), float("-inf"), device=inputs.device),
            diagonal=1,
        )
        residual = x = inputs
        x, _ = self.self_attention(x, x, x, attn_mask=causal_mask, need_weights=False)
        x = self.dropout(x)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        x = self.relu(self.feed_forward_1(x))
        x = self.feed_forward_2(x)
        x = self.dropout(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

In [ ]:
# PyTorch: keras.ops.* become torch.* ops. The reverse path reuses the token
# embedding matrix (weight) to project hidden states back to vocab logits.
class PositionalEmbedding(nn.Module):
    def __init__(self, sequence_length, input_dim, output_dim):
        super().__init__()
        self.token_embeddings = nn.Embedding(input_dim, output_dim)
        self.position_embeddings = nn.Embedding(sequence_length, output_dim)

    def forward(self, inputs, reverse=False):
        if reverse:
            return torch.matmul(inputs, self.token_embeddings.weight.t())
        positions = torch.cumsum(torch.ones_like(inputs), dim=-1) - 1
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

In [ ]:
# PyTorch: the functional keras.Model becomes an nn.Module that holds the shared
# PositionalEmbedding, an input LayerNorm, and the stack of decoders. The shared
# embedding is used both to embed inputs and (reverse=True) to produce logits.
# There is no mixed_float16 dtype policy here; use torch.autocast in training if
# desired.
vocab_size = tokenizer.vocab_size()
hidden_dim = 512
intermediate_dim = 2056
num_heads = 8
num_layers = 8


class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = PositionalEmbedding(sequence_length, vocab_size, hidden_dim)
        self.layernorm = nn.LayerNorm(hidden_dim)
        self.decoders = nn.ModuleList(
            [
                TransformerDecoder(hidden_dim, intermediate_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

    def forward(self, inputs):
        x = self.embedding(inputs)
        x = self.layernorm(x)
        for decoder in self.decoders:
            x = decoder(x)
        return self.embedding(x, reverse=True)


mini_gpt = MiniGPT()

#### Pretraining the model

In [ ]:
# PyTorch: a LearningRateSchedule becomes a plain callable returning a multiplier
# for torch.optim.lr_scheduler.LambdaLR. The base lr (2e-4) lives on the optimizer,
# so this returns the linear warmup scale in [0, 1].
class WarmupSchedule:
    def __init__(self):
        self.rate = 2e-4
        self.warmup_steps = 1_000.0

    def __call__(self, step):
        scale = min(step / self.warmup_steps, 1.0)
        return scale

In [ ]:
import matplotlib.pyplot as plt

schedule = WarmupSchedule()
x = range(0, 5_000, 100)
# PyTorch: the schedule returns a scale; multiply by the base rate to plot the lr.
y = [schedule.rate * schedule(step) for step in x]
plt.plot(x, y)
plt.xlabel("Train Step")
plt.ylabel("Learning Rate")
plt.show()

In [0]:
# ⚠️NOTE⚠️: If you can run the following with a Colab Pro GPU, we suggest you
# do so. This fit() call will take many hours on free tier GPUs. You can also
# reduce steps_per_epoch to try the code with a less trained model.

In [ ]:
# PyTorch: compile()/fit() become an explicit training loop. SparseCategorical-
# Crossentropy(from_logits=True) maps to nn.CrossEntropyLoss (logits + int targets);
# we flatten (batch, seq, vocab) to (batch*seq, vocab). The Adam optimizer carries
# the base learning rate and LambdaLR applies the warmup scale per step. A no_grad
# validation pass plays the role of validation_data.
num_epochs = 8
steps_per_epoch = num_train_batches // num_epochs
validation_steps = num_val_batches

device = "cuda" if torch.cuda.is_available() else "cpu"
mini_gpt.to(device)

optimizer = torch.optim.Adam(mini_gpt.parameters(), lr=schedule.rate)
lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=schedule)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(num_epochs):
    mini_gpt.train()
    train_iter = iter(train_loader)
    for step in range(steps_per_epoch):
        inputs, targets = next(train_iter)
        inputs, targets = inputs.to(device), targets.to(device)
        logits = mini_gpt(inputs)
        loss = loss_fn(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

    mini_gpt.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in val_batches[:validation_steps]:
            inputs, targets = inputs.to(device), targets.to(device)
            logits = mini_gpt(inputs)
            flat_logits = logits.reshape(-1, logits.shape[-1])
            flat_targets = targets.reshape(-1)
            val_loss += loss_fn(flat_logits, flat_targets).item()
            val_correct += (flat_logits.argmax(-1) == flat_targets).sum().item()
            val_total += flat_targets.numel()
    print(
        f"Epoch {epoch}: train loss {loss.item():.4f} "
        f"val loss {val_loss / len(val_batches[:validation_steps]):.4f} "
        f"val acc {val_correct / val_total:.4f}"
    )

#### Generative decoding

In [ ]:
# PyTorch: the generation loop runs under eval()/no_grad(). We feed the growing
# token list as a tensor and take the argmax of the last-position logits.
def generate(prompt, max_length=64):
    mini_gpt.eval()
    tokens = list(tokenizer.encode(prompt))
    prompt_length = len(tokens)
    for _ in range(max_length - prompt_length):
        with torch.no_grad():
            inputs = torch.tensor([tokens], dtype=torch.long, device=device)
            prediction = mini_gpt(inputs)
        prediction = prediction[0, -1]
        tokens.append(torch.argmax(prediction).item())
    return tokenizer.decode(tokens)

In [0]:
prompt = "A piece of advice"
generate(prompt)

In [ ]:
# PyTorch: there is no separate compiled vs. eager path like in Keras/JAX. This
# pre-allocates the token buffer and predicts position by position; for real speed
# you would wrap mini_gpt with torch.compile().
def compiled_generate(prompt, max_length=64):
    mini_gpt.eval()
    tokens = list(tokenizer.encode(prompt))
    prompt_length = len(tokens)
    tokens = tokens + [0] * (max_length - prompt_length)
    for i in range(prompt_length, max_length):
        with torch.no_grad():
            inputs = torch.tensor([tokens], dtype=torch.long, device=device)
            prediction = mini_gpt(inputs)
        prediction = prediction[0, i - 1]
        tokens[i] = torch.argmax(prediction).item()
    return tokenizer.decode(tokens)

In [0]:
import timeit
tries = 10
timeit.timeit(lambda: compiled_generate(prompt), number=tries) / tries

#### Sampling strategies

In [ ]:
# PyTorch: same loop, but a sampling function chooses the next token from the
# last-position logits (returned as a Python int).
def compiled_generate(prompt, sample_fn, max_length=64):
    mini_gpt.eval()
    tokens = list(tokenizer.encode(prompt))
    prompt_length = len(tokens)
    tokens = tokens + [0] * (max_length - prompt_length)
    for i in range(prompt_length, max_length):
        with torch.no_grad():
            inputs = torch.tensor([tokens], dtype=torch.long, device=device)
            prediction = mini_gpt(inputs)
        prediction = prediction[0, i - 1]
        next_token = sample_fn(prediction)
        tokens[i] = int(next_token)
    return tokenizer.decode(tokens)

In [ ]:
def greedy_search(preds):
    return torch.argmax(preds)


compiled_generate(prompt, greedy_search)

In [ ]:
# PyTorch: keras.random.categorical -> torch.multinomial over softmax probs.
def random_sample(preds, temperature=1.0):
    preds = preds / temperature
    probs = torch.softmax(preds, dim=-1)
    return torch.multinomial(probs, num_samples=1)[0]

In [0]:
compiled_generate(prompt, random_sample)

In [0]:
from functools import partial
compiled_generate(prompt, partial(random_sample, temperature=2.0))

In [0]:
compiled_generate(prompt, partial(random_sample, temperature=0.8))

In [0]:
compiled_generate(prompt, partial(random_sample, temperature=0.2))

In [ ]:
# PyTorch: ops.top_k -> torch.topk; sample within the top-k logits and gather the
# original vocab id with the sampled index.
def top_k(preds, k=5, temperature=1.0):
    preds = preds / temperature
    top_preds, top_indices = torch.topk(preds, k=k, sorted=False)
    probs = torch.softmax(top_preds, dim=-1)
    choice = torch.multinomial(probs, num_samples=1)
    return top_indices[choice][0]

In [0]:
compiled_generate(prompt, partial(top_k, k=5))

In [0]:
compiled_generate(prompt, partial(top_k, k=20))

In [0]:
compiled_generate(prompt, partial(top_k, k=5, temperature=0.5))

### Using a pretrained LLM

#### Text generation with the Gemma model

In [ ]:
# PyTorch: GPT-2 weights download from the HuggingFace Hub without any login.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# PyTorch: keras_hub's Gemma preset is replaced by HuggingFace transformers GPT-2,
# the closest widely available pretrained decoder LLM. (Gemma weights are gated
# and Keras-specific; GPT-2 preserves the chapter's intent of running a pretrained
# causal LM.)
hf_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gemma_lm = GPT2LMHeadModel.from_pretrained("gpt2")
gemma_lm.eval()
gemma_lm.to(device)

In [ ]:
# PyTorch: print the module (HuggingFace models have no Keras-style summary()).
print(gemma_lm)

In [ ]:
# PyTorch: gemma_lm.compile(sampler=...) + .generate become model.generate. Greedy
# decoding is do_sample=False. We wrap tokenize -> generate -> decode in a helper.
def hf_generate(prompt, max_length=40, do_sample=False):
    input_ids = hf_tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        output_ids = gemma_lm.generate(
            input_ids,
            max_length=max_length,
            do_sample=do_sample,
            pad_token_id=hf_tokenizer.eos_token_id,
        )
    return hf_tokenizer.decode(output_ids[0], skip_special_tokens=True)


hf_generate("A piece of advice", max_length=40)

In [ ]:
hf_generate("How can I make brownies?", max_length=40)

In [ ]:
hf_generate(
    "The following brownie recipe is easy to make in just a few "
    "steps.\n\nYou can start by",
    max_length=40,
)

In [ ]:
hf_generate(
    "Tell me about the 542nd president of the United States.",
    max_length=40,
)

#### Instruction fine-tuning

In [ ]:
import json
import urllib.request

# PyTorch: keras.utils.get_file -> urllib download of the same Dolly-15k jsonl.
PROMPT_TEMPLATE = """"[instruction]\n{}[end]\n[response]\n"""
RESPONSE_TEMPLATE = """{}[end]"""

dataset_path = "databricks-dolly-15k.jsonl"
urllib.request.urlretrieve(
    "https://hf.co/datasets/databricks/databricks-dolly-15k/"
    "resolve/main/databricks-dolly-15k.jsonl",
    dataset_path,
)
data = {"prompts": [], "responses": []}
with open(dataset_path) as file:
    for line in file:
        features = json.loads(line)
        if features["context"]:
            continue
        data["prompts"].append(PROMPT_TEMPLATE.format(features["instruction"]))
        data["responses"].append(RESPONSE_TEMPLATE.format(features["response"]))

In [0]:
data["prompts"][0]

In [0]:
data["responses"][0]

In [ ]:
import random

# PyTorch: build (prompt, response) pairs, shuffle, and hold out 100 for validation
# (replacing tf.data shuffle/batch/take/skip).
examples = list(zip(data["prompts"], data["responses"]))
random.Random(0).shuffle(examples)
val_examples = examples[:100]
train_examples = examples[100:]

In [ ]:
# PyTorch: keras_hub's preprocessor is replaced by tokenizing prompt+response with
# the GPT-2 tokenizer and padding to a fixed length. token_ids is the input,
# padding_mask marks real tokens, y is the next-token target (shifted), and
# sample_weight masks out the prompt + padding so loss is only on the response.
sequence_length = 512


def preprocess(batch):
    token_ids, padding_masks, labels, sample_weights = [], [], [], []
    for prompt, response in batch:
        prompt_ids = hf_tokenizer.encode(prompt)
        response_ids = hf_tokenizer.encode(response)
        ids = (prompt_ids + response_ids)[:sequence_length]
        mask = [1] * len(ids)
        # weight only the response tokens
        weight = [0] * min(len(prompt_ids), len(ids))
        weight += [1] * (len(ids) - len(weight))
        pad = sequence_length - len(ids)
        ids = ids + [0] * pad
        mask = mask + [0] * pad
        weight = weight + [0] * pad
        token_ids.append(ids)
        padding_masks.append(mask)
        sample_weights.append(weight)
    token_ids = torch.tensor(token_ids, dtype=torch.long)
    padding_mask = torch.tensor(padding_masks, dtype=torch.long)
    sample_weight = torch.tensor(sample_weights, dtype=torch.float)
    # next-token targets (shift left by one)
    y = torch.roll(token_ids, shifts=-1, dims=1)
    return {"token_ids": token_ids, "padding_mask": padding_mask}, y, sample_weight


batch = train_examples[:2]
x, y, sample_weight = preprocess(batch)
x["token_ids"].shape

In [ ]:
x["padding_mask"].shape

In [ ]:
y.shape

In [ ]:
sample_weight.shape

In [ ]:
x["token_ids"][0, :5], y[0, :5]

#### Low-Rank Adaptation (LoRA)

In [ ]:
# PyTorch: keras_hub's backbone.enable_lora is replaced by HuggingFace PEFT LoRA.
# This wraps GPT-2's attention projections with low-rank adapters (rank=8).
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(r=8, lora_alpha=8, target_modules=["c_attn"], task_type="CAUSAL_LM")
gemma_lm = get_peft_model(gemma_lm, lora_config)

In [ ]:
# PyTorch: PEFT exposes the trainable (LoRA) vs. frozen parameter counts.
gemma_lm.print_trainable_parameters()

In [ ]:
# PyTorch: compile()/fit() become a training loop. Loss is CrossEntropyLoss on the
# next-token logits, weighted by sample_weight so only response tokens count
# (the Keras weighted_metrics / SparseCategoricalCrossentropy equivalent).
gemma_lm.train()
gemma_lm.to(device)
optimizer = torch.optim.Adam(gemma_lm.parameters(), lr=5e-5)


def loss_on_batch(examples):
    x, y, sample_weight = preprocess(examples)
    token_ids = x["token_ids"].to(device)
    y = y.to(device)
    sample_weight = sample_weight.to(device)
    logits = gemma_lm(input_ids=token_ids).logits
    per_token = nn.functional.cross_entropy(
        logits.reshape(-1, logits.shape[-1]), y.reshape(-1), reduction="none"
    )
    per_token = per_token * sample_weight.reshape(-1)
    return per_token.sum() / sample_weight.sum().clamp(min=1.0)


batch_size = 2
for epoch in range(1):
    random.Random(epoch).shuffle(train_examples)
    for i in range(0, len(train_examples), batch_size):
        loss = loss_on_batch(train_examples[i : i + batch_size])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        val_loss = sum(
            loss_on_batch(val_examples[i : i + batch_size]).item()
            for i in range(0, len(val_examples), batch_size)
        )
    print(f"Epoch {epoch}: train loss {loss.item():.4f} val loss {val_loss:.4f}")

In [ ]:
gemma_lm.eval()


def instruct_generate(prompt, max_length=512):
    input_ids = hf_tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        output_ids = gemma_lm.generate(
            input_ids,
            max_length=max_length,
            do_sample=False,
            pad_token_id=hf_tokenizer.eos_token_id,
        )
    return hf_tokenizer.decode(output_ids[0], skip_special_tokens=True)


instruct_generate(
    "[instruction]\nHow can I make brownies?[end]\n"
    "[response]\n",
    max_length=512,
)

In [ ]:
instruct_generate(
    "[instruction]\nWhat is a proper noun?[end]\n"
    "[response]\n",
    max_length=512,
)

In [ ]:
instruct_generate(
    "[instruction]\nWho is the 542nd president of the United States?[end]\n"
    "[response]\n",
    max_length=512,
)

### Going further with LLMs

#### Reinforcement Learning with Human Feedback (RLHF)

##### Using a chatbot trained with RLHF

In [ ]:
# PyTorch: no runtime restart / kagglehub login is needed. Re-import what we use.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# PyTorch: the gated Gemma-3-instruct-4b preset is replaced by an openly available
# instruction-tuned chat model (TinyLlama-Chat) so the RLHF/chat section still runs
# without authentication. Swap in any HF chat model you prefer.
chat_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
chat_tokenizer = AutoTokenizer.from_pretrained(chat_model_id)
gemma_lm = AutoModelForCausalLM.from_pretrained(
    chat_model_id, torch_dtype=torch.bfloat16
)
gemma_lm.eval()
gemma_lm.to(device)

In [ ]:
# PyTorch: HF chat models carry their own chat template, but we keep an explicit
# template string to mirror the book. We define a helper that runs generate().
PROMPT_TEMPLATE = """<|user|>
{}</s>
<|assistant|>
"""


def chat_generate(prompt, max_length=512, do_sample=False):
    input_ids = chat_tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        output_ids = gemma_lm.generate(
            input_ids,
            max_new_tokens=max_length,
            do_sample=do_sample,
            pad_token_id=chat_tokenizer.eos_token_id,
        )
    return chat_tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
prompt = "Why can't you assign values in Jax tensors? Be brief!"
chat_generate(PROMPT_TEMPLATE.format(prompt), max_length=512)

In [ ]:
prompt = "Who is the 542nd president of the United States?"
chat_generate(PROMPT_TEMPLATE.format(prompt), max_length=512)

#### Multimodal LLMs

In [ ]:
import urllib.request
import matplotlib.pyplot as plt
from PIL import Image

# PyTorch: keras.utils.get_file/load_img -> urllib + PIL.
image_url = (
    "https://github.com/mattdangerw/keras-nlp-scripts/"
    "blob/main/learned-python.png?raw=true"
)
image_path = "learned-python.png"
urllib.request.urlretrieve(image_url, image_path)

image = np.array(Image.open(image_path).convert("RGB"))
plt.axis("off")
plt.imshow(image)
plt.show()

In [ ]:
# PyTorch: the substitute TinyLlama chat model is text-only, so true multimodal
# (image) prompting is not available. To run the chapter's vision section, load a
# HuggingFace vision-language model (e.g. llava-hf/llava-1.5-7b-hf) and pass the
# image through its processor. Here we just describe the intended call.
prompt = "What is going on in this image? Be concise!"
chat_generate(PROMPT_TEMPLATE.format(prompt), max_length=512)

In [ ]:
prompt = "What is the snake wearing?"
chat_generate(PROMPT_TEMPLATE.format(prompt), max_length=512)

##### Foundation models

#### Retrieval Augmented Generation (RAG)

#### "Reasoning" models

In [ ]:
prompt = """Judy wrote a 2-page letter to 3 friends twice a week for 3 months.
How many letters did she write?
Be brief, and add "ANSWER:" before your final answer."""

# PyTorch: gemma_lm.compile(sampler="random") -> sampling at generate time
# (do_sample=True), set below in the generate calls.

In [ ]:
chat_generate(PROMPT_TEMPLATE.format(prompt), do_sample=True)

In [ ]:
chat_generate(PROMPT_TEMPLATE.format(prompt), do_sample=True)

### Where are LLMs heading next?